**Difficulty:** ⭐⭐

# Analyse `data/hdb_resale_prices.csv` here!

This data is already cleaned because it was actually taken from publicly available records from the government (https://data.gov.sg/datasets/d_8b84c4ee58e3cfc0ece0d773c8ca6abc/view). Using `csv.DictReader` would be very helpful here to keep track of the various headers!

## Price per Square Metre

When looking at housing prices, the absolute `resale_price` doesn't actually tell the full story because the bigger the house, the more expensive it is. Thus looking at **price per square metre** allows for a more "apples-to-apples" comparison.

$ Price\_Per\_Sqm = resale\_price / floor\_area\_sqm$

Create a new column called `price_per_sqm` and save it in a new file called `hdb_resale_prices_v2.csv` - you should be using that file from now on.

**Difficulty levels:** ⭐ Beginner · ⭐⭐ Intermediate · ⭐⭐⭐ Advanced

In [2]:
import csv

input_file = "hdb_resale_prices.csv"
output_file = "hdb_resale_prices_v2.csv"

# If you get a FileNotFoundError
# Download hdb_resale_prices.csv from the data folder in the drive
# Drag and drop the file into the 'Files' tab on the left side of the notebook
# Run the code again
with open(input_file) as infile, open(output_file, "w") as outfile:

    reader = csv.DictReader(infile)
    fieldnames = reader.fieldnames + ["price_per_sqm"]

    writer = csv.DictWriter(outfile, fieldnames=fieldnames)
    writer.writeheader()

    for row in reader:
        resale_price = float(row["resale_price"])
        floor_area_sqm = float(row["floor_area_sqm"])
        row["price_per_sqm"] = round(resale_price / floor_area_sqm, 2)
        writer.writerow(row)

print(f"Done! Saved to {output_file}")

Done! Saved to hdb_resale_prices_v2.csv


**Difficulty:** ⭐⭐

## Expensive Neighbourhoods

We can find out which neighbourhoods are the most expensive. Calculate the average `price_per_sqm` for each `town`.

*Hint: Use a dictionary where the keys are town names and values are a list of prices*

In [3]:
import csv

input_file = "hdb_resale_prices_v2.csv"

# Build a dict of town -> list of price_per_sqm values
town_prices = {}

with open(input_file) as f:
    reader = csv.DictReader(f)

    for row in reader:
        town = row["town"]
        price_per_sqm = float(row["price_per_sqm"])

        if town not in town_prices:
            town_prices[town] = []

        town_prices[town].append(price_per_sqm)

# Calculate average price_per_sqm for each town
town_avg = {}
for town, prices in town_prices.items():
    town_avg[town] = round(sum(prices) / len(prices), 2)

# Sort by average price descending
town_avg_sorted = sorted(town_avg.items(), key=lambda x: x[1], reverse=True)

print(f"{'Town':<30} {'Avg Price/sqm':>15}")
print("-" * 46)
for town, avg in town_avg_sorted:
    print(f"{town:<30} {avg:>15,.2f}")

Town                             Avg Price/sqm
----------------------------------------------
CENTRAL AREA                          8,252.61
QUEENSTOWN                            7,620.57
BUKIT MERAH                           7,267.12
BUKIT TIMAH                           7,041.34
KALLANG/WHAMPOA                       6,786.95
BISHAN                                6,607.91
TOA PAYOH                             6,477.34
MARINE PARADE                         6,399.83
CLEMENTI                              6,383.89
GEYLANG                               6,028.98
PUNGGOL                               5,781.31
SERANGOON                             5,602.77
ANG MO KIO                            5,569.34
TAMPINES                              5,509.93
SENGKANG                              5,417.39
BUKIT BATOK                           5,415.25
BEDOK                                 5,350.83
SEMBAWANG                             5,297.76
HOUGANG                               5,293.94
BUKIT PANJANG

**Difficulty:** ⭐⭐

## Lease Decay

We can explore the relationship between the age of a flat and its price. Find the 10 oldest flats (based on `lease_commence_date`) that were sold for over $800,000.

In [4]:
import csv

# Filter flats sold for over $800,000
expensive_flats = []

with open(input_file, newline="", encoding="utf-8") as infile:
    reader = csv.DictReader(infile)

    for row in reader:
        resale_price = float(row["resale_price"])

        if resale_price > 800000:
            expensive_flats.append(row)

# Sort by lease_commence_date ascending (oldest first)
expensive_flats.sort(key=lambda x: int(x["lease_commence_date"]))

# Take the 10 oldest
top_10_oldest = expensive_flats[:10]

# Display results
cols = ["town", "flat_type", "lease_commence_date", "resale_price", "price_per_sqm"]
header = f"{'Town':<25} {'Flat Type':<20} {'Lease Start':>12} {'Resale Price':>14} {'Price/sqm':>10}"
print(header)
print("-" * 85)

for row in top_10_oldest:
    print(
        f"{row['town']:<25} {row['flat_type']:<20} "
        f"{row['lease_commence_date']:>12} "
        f"{float(row['resale_price']):>14,.0f} "
        f"{float(row['price_per_sqm']):>10,.2f}"
    )

Town                      Flat Type             Lease Start   Resale Price  Price/sqm
-------------------------------------------------------------------------------------
QUEENSTOWN                3 ROOM                       1968        870,000   8,055.56
QUEENSTOWN                3 ROOM                       1968        888,889  10,582.01
QUEENSTOWN                3 ROOM                       1968        888,000  10,206.90
QUEENSTOWN                4 ROOM                       1968        858,000   8,580.00
QUEENSTOWN                3 ROOM                       1968        850,000   9,770.11
QUEENSTOWN                3 ROOM                       1968        830,000   9,540.23
QUEENSTOWN                4 ROOM                       1968        928,000   7,083.97
QUEENSTOWN                3 ROOM                       1968        855,000   8,382.35
QUEENSTOWN                4 ROOM                       1968        975,000   7,276.12
QUEENSTOWN                3 ROOM                      

**Difficulty:** ⭐⭐

## Storey Premium

Does living on a higher floor actually cost more?

Compare the average `price_per_sqm` of flats in the `01 TO 03` storey range against those in the `10 TO 12` range within the same town.

In [5]:
import csv

LOW_STOREY = "01 TO 03"
HIGH_STOREY = "10 TO 12"

# Dict structure: { town: { "low": [...], "high": [...] } }
town_storey_prices = {}

with open(input_file) as f:
    reader = csv.DictReader(f)

    for row in reader:
        storey_range = row["storey_range"].strip()

        if storey_range not in (LOW_STOREY, HIGH_STOREY):
            continue

        town = row["town"]
        price_per_sqm = float(row["price_per_sqm"])

        if town not in town_storey_prices:
            town_storey_prices[town] = {"low": [], "high": []}

        if storey_range == LOW_STOREY:
            town_storey_prices[town]["low"].append(price_per_sqm)
        else:
            town_storey_prices[town]["high"].append(price_per_sqm)

# Calculate averages and premium
results = []

for town, prices in town_storey_prices.items():
    if not prices["low"] or not prices["high"]:
        continue  # skip towns missing data for either storey range

    avg_low = sum(prices["low"]) / len(prices["low"])
    avg_high = sum(prices["high"]) / len(prices["high"])
    premium = avg_high - avg_low

    results.append((town, avg_low, avg_high, premium))

# Sort by premium descending
results.sort(key=lambda x: x[3], reverse=True)

# Display results
print(f"{'Town':<25} {'Avg 01-03':>10} {'Avg 10-12':>10} {'Premium':>10}")
print("-" * 60)

for town, avg_low, avg_high, premium in results:
    print(f"{town:<25} {avg_low:>10,.2f} {avg_high:>10,.2f} {premium:>+10,.2f}")

Town                       Avg 01-03  Avg 10-12    Premium
------------------------------------------------------------
CENTRAL AREA                5,834.14   7,271.40  +1,437.26
CLEMENTI                    5,061.82   6,020.15    +958.33
GEYLANG                     5,389.98   6,232.52    +842.53
QUEENSTOWN                  6,188.30   7,015.62    +827.32
MARINE PARADE               5,770.64   6,528.03    +757.39
SERANGOON                   5,182.25   5,923.40    +741.15
TOA PAYOH                   5,377.89   6,103.06    +725.16
TAMPINES                    5,078.03   5,753.96    +675.93
BISHAN                      6,037.67   6,674.62    +636.95
BUKIT BATOK                 4,854.31   5,489.56    +635.25
SENGKANG                    4,968.28   5,538.92    +570.65
BUKIT MERAH                 6,292.23   6,839.32    +547.09
ANG MO KIO                  4,891.91   5,431.64    +539.72
BUKIT PANJANG               4,469.65   4,989.87    +520.23
SEMBAWANG                   4,987.46   5,495.12    +50

**Difficulty:** ⭐⭐

## Market Trend

Since the data starts from 2017, we can see how prices changed over time.  
Calculate the average `price_per_sqm` for each year (2017, 2018, etc.) to see if the market is trending up or down.

In [7]:
import csv

# Dict structure: { year: [...prices...] }
yearly_prices = {}

with open(input_file) as f:
    reader = csv.DictReader(f)

    for row in reader:
        year = row["month"][:4]  # Extract year from "YYYY-MM"
        price_per_sqm = float(row["price_per_sqm"])

        if year not in yearly_prices:
            yearly_prices[year] = []

        yearly_prices[year].append(price_per_sqm)

# Calculate average per year, sorted chronologically
results = []

for year, prices in yearly_prices.items():
    avg = round(sum(prices) / len(prices), 2)
    results.append((year, avg, len(prices)))

results.sort(key=lambda x: x[0])

# Display results with year-on-year change
print(f"{'Year':<8} {'Avg Price/sqm':>14} {'Transactions':>14} {'YoY Change':>12}")
print("-" * 52)

prev_avg = None
for year, avg, count in results:
    if prev_avg is None:
        change_str = "         -"
    else:
        change = avg - prev_avg
        change_str = f"{change:>+10,.2f}"

    print(f"{year:<8} {avg:>14,.2f} {count:>14,} {change_str:>12}")
    prev_avg = avg

Year      Avg Price/sqm   Transactions   YoY Change
----------------------------------------------------
2017           4,578.88         20,509            -
2018           4,509.67         21,561       -69.21
2019           4,476.80         22,186       -32.87
2020           4,669.83         23,333      +193.03
2021           5,248.68         29,087      +578.85
2022           5,723.79         26,720      +475.11
2023           6,071.26         25,754      +347.47
2024           6,493.36         27,832      +422.10
2025           6,953.71         25,086      +460.35
2026           7,001.24          6,960       +47.53


## Explore for Yourself!

What else can you find out about this dataset?

---

# 🏠 Homework

*Take these home and attempt them before the next session. Difficulty markers are shown for each exercise.*

## Fix-it 🔨 ⭐

This never closes the file properly and doesn't handle errors. Rewrite it using `with` and `try/except`.

In [ ]:
f = open('data.txt', 'r')
print(f.read())


## Predict 🔮 ⭐

A file `nums.txt` contains `1\n2\n3\n`. What is printed?

In [ ]:
with open('nums.txt') as f:
    lines = f.readlines()
print(len(lines))
# Prediction:


## Fill-in-the-blanks ✏️ ⭐⭐

Complete the CSV writer to save a list of rows.

In [ ]:
import csv
rows = [['name','score'], ['Alex', 90]]
with open('out.csv', 'w', newline='') as f:
    writer = csv._______(f)
    writer._______(rows)


---

# 🤝 Group Activity

**The Log Merger**

In pairs, merge two log files into one sorted log. Each line of `log_a.txt` and `log_b.txt` starts with a timestamp. Read both, combine the lines, sort them by timestamp, and write the result to `merged_log.txt`. One partner reads the files, the other writes the merged output. Use `try`/`except` to handle a missing file.

In [ ]:
# write code here


---

# 📊 Differentiated Practice

Choose the level that challenges you most — try to work your way up!

## Beginner ⭐

Write a program that asks the user for 3 lines of text and writes them to a file called `notes.txt`.

In [ ]:
# write code here


## Intermediate ⭐⭐

Read `notes.txt` and print how many words it contains. Wrap the file open in `try`/`except` to handle a missing file.

> 💡 *Hint:* Use `f.read().split()` to get a list of words.

In [ ]:
# write code here


## Advanced ⭐⭐⭐

Read a CSV of student grades. Compute each student's average and write a new CSV with columns `name, average`.

In [ ]:
# write code here
